# Извлечение признаков tsfresh

Из готовых секундных рядов в `analysis/tsfresh/results/second_series.csv.gz` получаем один вектор признаков на минуту. `windows.csv` задаёт порядок строк по `window_id`.

Результат сохраняется в `analysis/tsfresh/results/comprehensive_features.csv.gz`. Его читает [EDA_tsfresh.ipynb](EDA_tsfresh.ipynb). Для запуска выберите ядро окружения `.venv-tsfresh` с tsfresh 0.21.1.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import warnings

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from tsfresh import __version__ as tsfresh_version, extract_features
from tsfresh.feature_extraction import ComprehensiveFCParameters

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "analysis/tsfresh/run_analysis.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Запустите ноутбук из каталога проекта phd.")

results = root / "analysis/tsfresh/results"
series_path = results / "second_series.csv.gz"
windows_path = results / "windows.csv"
features_path = results / "comprehensive_features.csv.gz"
channels = ["tcp_flows", "udp_flows", "packets", "bytes", "syn", "rst"]

for path in (series_path, windows_path):
    if not path.is_file():
        raise FileNotFoundError(f"Нет подготовленных данных: {path}. См. analysis/tsfresh/README.md.")

window_ids = pd.Index(pd.read_csv(windows_path, usecols=["window_id"])["window_id"])
series = pd.read_csv(
    series_path,
    usecols=["window_id", "time", *channels],
    parse_dates=["time"],
    dtype={channel: "float64" for channel in channels},
).sort_values(["window_id", "time"])

samples_per_window = series.groupby("window_id").size()
if (window_ids.empty or not window_ids.is_unique
        or not samples_per_window.index.equals(window_ids.sort_values())
        or not samples_per_window.eq(60).all()
        or series.duplicated(["window_id", "time"]).any()
        or series["time"].isna().any()):
    raise ValueError("Секундные ряды должны содержать 60 разных отсчётов на каждое окно из windows.csv.")
if not np.isfinite(series[channels].to_numpy()).all():
    raise ValueError("Секундные ряды содержат нечисловые или бесконечные значения.")

print(f"tsfresh {tsfresh_version}, окон: {len(window_ids)}, каналов: {len(channels)}")

tsfresh 0.21.1, окон: 2442, каналов: 6


## Расчёт

`ComprehensiveFCParameters` задаёт полный набор калькуляторов. На вход идут шесть рядов в `float64`, время задаёт порядок отсчётов. Неопределённые значения признаков остаются пропусками.

In [2]:
started = perf_counter()
with warnings.catch_warnings(), threadpool_limits(limits=1):
    warnings.simplefilter("ignore", RuntimeWarning)
    features = extract_features(
        series,
        column_id="window_id",
        column_sort="time",
        default_fc_parameters=ComprehensiveFCParameters(),
        impute_function=None,
        n_jobs=0,
    )

if not features.index.is_unique or not features.index.sort_values().equals(window_ids.sort_values()):
    raise ValueError("Извлечённые признаки и windows.csv содержат разные окна.")
features = features.reindex(window_ids).replace([np.inf, -np.inf], np.nan)
print(f"Окон: {len(features)}, признаков: {features.shape[1]}. Время: {perf_counter() - started:.1f} с.")

Окон: 2442, признаков: 4698. Время: 236.2 с.


## Сохранение

В CSV одна строка на окно. Первый столбец `window_id` связывает признаки с разметкой в EDA.

In [3]:
with TemporaryDirectory(dir=results) as temp_dir:
    temp_path = Path(temp_dir) / features_path.name
    features.to_csv(temp_path, index_label="window_id")
    temp_path.replace(features_path)

print(f"Файл: {features_path.relative_to(root)}")
print(f"Размер: {features_path.stat().st_size / 1024**2:.1f} МБ")

Файл: analysis/tsfresh/results/comprehensive_features.csv.gz
Размер: 43.5 МБ
